# Sentiment classification with leakage-safe model selection

The project builds a TF-IDF logistic-regression baseline for NLTK movie reviews. The held-out test set is untouched during hyperparameter selection. Evaluation includes macro-F1, Matthews correlation, probability calibration and an explicit error table.

In [ ]:
import nltk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

nltk.download("movie_reviews", quiet=True)
from nltk.corpus import movie_reviews
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, ConfusionMatrixDisplay,
                             matthews_corrcoef, brier_score_loss)
from sklearn.calibration import CalibratedClassifierCV, CalibrationDisplay
from nlp_portfolio.evaluation import expected_calibration_error

texts = np.array([" ".join(movie_reviews.words(fid)) for fid in movie_reviews.fileids()], dtype=object)
labels = np.array([movie_reviews.categories(fid)[0] for fid in movie_reviews.fileids()])
pd.Series(labels).value_counts().rename_axis("label").to_frame("count")

In [ ]:
train_idx, test_idx = train_test_split(
    np.arange(len(labels)), test_size=0.20, random_state=42, stratify=labels
)
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
    ("classifier", LogisticRegression(max_iter=1500, random_state=42)),
])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
search = GridSearchCV(
    pipeline,
    {"classifier__C": [0.1, 0.3, 1, 3, 10], "tfidf__max_df": [0.9, 1.0]},
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    return_train_score=True,
)
search.fit(texts[train_idx].tolist(), labels[train_idx])
print("best parameters:", search.best_params_)
print("cross-validated macro-F1:", round(search.best_score_, 3))

Cross-validation is applied to the complete pipeline, so the vocabulary and inverse-document-frequency values are re-estimated inside every training fold. This prevents validation documents from influencing the feature representation.

In [ ]:
predictions = search.predict(texts[test_idx].tolist())
print(classification_report(labels[test_idx], predictions, digits=3))
print("Matthews correlation:", round(matthews_corrcoef(labels[test_idx], predictions), 3))
ConfusionMatrixDisplay.from_predictions(labels[test_idx], predictions, cmap="Blues")
plt.title("Held-out confusion matrix")
plt.show()

## Probability calibration

Accuracy alone does not tell us whether a reported probability is meaningful. Sigmoid calibration is fitted only on training folds. Brier score measures squared probability error and ECE summarizes the gap between confidence and observed frequency.

In [ ]:
calibrated = CalibratedClassifierCV(search.best_estimator_, method="sigmoid", cv=5)
calibrated.fit(texts[train_idx].tolist(), labels[train_idx])
positive = list(calibrated.classes_).index("pos")
probability = calibrated.predict_proba(texts[test_idx].tolist())[:, positive]
y_binary = (labels[test_idx] == "pos").astype(int)
print("Brier score:", round(brier_score_loss(y_binary, probability), 4))
print("ECE:", round(expected_calibration_error(y_binary, probability, bins=10), 4))
CalibrationDisplay.from_predictions(y_binary, probability, n_bins=10)
plt.title("Held-out probability calibration")
plt.show()

## What the linear model learned

Coefficients identify influential n-grams, conditional on correlated features. They are useful for auditing shortcuts but should not be read as causal effects.

In [ ]:
vectorizer = search.best_estimator_.named_steps["tfidf"]
classifier = search.best_estimator_.named_steps["classifier"]
features = vectorizer.get_feature_names_out()
weights = classifier.coef_[0]
top = 15
coefficient_table = pd.DataFrame({
    "negative": features[np.argsort(weights)[:top]],
    "positive": features[np.argsort(weights)[-top:][::-1]],
})
coefficient_table

In [ ]:
confidence = np.maximum(probability, 1 - probability)
errors = pd.DataFrame({
    "text": texts[test_idx], "gold": labels[test_idx], "predicted": predictions,
    "confidence": confidence,
})
errors = errors[errors.gold != errors.predicted].sort_values("confidence", ascending=False)
errors.head(10)

The highest-confidence mistakes are the most informative audit cases: mixed sentiment, quoted opinions and plot descriptions often violate the bag-of-ngrams assumption. A stronger comparison would repeat the complete split and tuning procedure across several random seeds.